# 딥러닝 프로젝트: PyTorch로 건강 데이터 기반 식단 추천 분류 모델 만들기

---

## 1. 데이터 불러오기

- CSV 파일을 DataFrame으로 불러옵니다.
- `df.head()`, `df.shape`, `df.info()`를 사용해서 데이터의 구조를 파악해 봅니다.

In [ ]:
# 필요한 라이브러리 불러오기
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# PyTorch 관련 라이브러리
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
# 전처리 및 평가 관련 라이브러리
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# CSV 파일 불러오기
df = pd.read_csv('https://bakey-api.codeit.kr/api/files/resource?root=static&seqId=14809&version=1&directory=/diet_recommendations.csv&name=diet_recommendations.csv')

# 데이터 일부 확인
df.head()

,Patient_ID,Age,Gender,Weight_kg,Height_cm,Physical_Activity_Level,Daily_Caloric_Intake,Cholesterol_mg/dL,Blood_Pressure_mmHg,Glucose_mg/dL,Dietary_Restrictions,Allergies,Preferred_Cuisine,Weekly_Exercise_Hours,Adherence_to_Diet_Plan,Dietary_Nutrient_Imbalance_Score,Diet_Recommendation
0,P0001,56,Male,114.1,190.0,Moderate,3140,212.1,153.8,125.6,Low_Sugar,Gluten,Indian,4.94,1.0,1.7,Low_Sodium
1,P0002,69,Female,66.1,163.0,Sedentary,1801,183.5,140.4,129.6,NaN,Peanuts,Italian,1.74,1.0,0.8,Balanced
2,P0003,46,Female,58.0,159.2,Sedentary,1869,245.8,112.4,91.6,Low_Sugar,Peanuts,Chinese,2.53,1.0,4.5,Balanced
3,P0004,32,Female,84.5,163.9,Sedentary,2094,147.3,146.1,129.3,Low_Sodium,NaN,Indian,2.11,1.0,0.5,Balanced
4,P0005,60,Female,79.3,169.1,Moderate,2222,245.0,145.8,128.6,Low_Sugar,Peanuts,Chinese,6.00,1.0,3.8,Balanced


- Patient_ID, Age, Gender, Weight_kg, Height_cm 등 환자의 기본 정보와 건강 지표, 생활 습관 정보, 그리고 추천 식단(Diet_Recommendation)이 포함되어 있습니다.

In [ ]:
# 데이터 크기 확인
print(df.shape)

(5000, 17)


- 총 5,000개의 환자 데이터와 17개의 컬럼으로 구성되어 있습니다.

In [ ]:
# 데이터 정보 확인
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 17 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Patient_ID                        5000 non-null   object 
 1   Age                               5000 non-null   int64  
 2   Gender                            5000 non-null   object 
 3   Weight_kg                         5000 non-null   float64
 4   Height_cm                         5000 non-null   float64
 5   Physical_Activity_Level           5000 non-null   object 
 6   Daily_Caloric_Intake              5000 non-null   int64  
 7   Cholesterol_mg/dL                 5000 non-null   float64
 8   Blood_Pressure_mmHg               5000 non-null   float64
 9   Glucose_mg/dL                     5000 non-null   float64
 10  Dietary_Restrictions              2915 non-null   object 
 11  Allergies                         3286 non-null   object 
 12  Prefer

- 수치형 데이터와 범주형 데이터가 혼합되어 있으며, Dietary_Restrictions와 Allergies 컬럼에는 결측값이 존재합니다.

## 2. 문제 정의 및 타깃 설정

- 이번 프로젝트의 목표는 환자의 건강 상태와 생활 습관 정보를 바탕으로 적합한 식단을 자동으로 추천하는 분류 모델을 PyTorch로 만드는 것입니다.
- 타깃 변수는 `Diet_Recommendation`이며, 이 컬럼에는 각 환자에게 추천된 식단 유형이 저장되어 있습니다.
- 타깃 변수의 분포를 확인하여 어떤 식단 유형들이 있는지, 각 클래스가 얼마나 균형 있게 분포되어 있는지 살펴봅니다.

In [ ]:
# 타깃 변수 분포 확인
df["Diet_Recommendation"].value_counts()

,count
Diet_Recommendation,
Balanced,2794
Low_Sodium,1229
Low_Carb,977


- Balanced, Low_Sodium, Low_Carb 세 가지 식단 유형이 있습니다. 딥러닝 모델을 통해 이 세 클래스를 분류해 볼 거예요.

## 3. 파생 변수 만들기

- BMI, BMI_Category, Age_Group, 고위험 건강 지표, Metabolic_Risk_Score 등을 추가하여 모델이 더 풍부한 정보를 학습할 수 있게 합니다.

In [ ]:
# BMI 계산
df["BMI"] = df["Weight_kg"] / (df["Height_cm"] / 100) ** 2

In [ ]:
# BMI_Category 생성
def categorize_bmi(bmi):
    if bmi < 18.5:
        return "Underweight"
    elif bmi < 25:
        return "Normal"
    elif bmi < 30:
        return "Overweight"
    else:
        return "Obese"

df["BMI_Category"] = df["BMI"].apply(categorize_bmi)

In [ ]:
# Age_Group 생성
bins = [18, 35, 50, 65, 80]
labels = ["18_34", "35_49", "50_64", "65_79"]
df["Age_Group"] = pd.cut(df["Age"], bins=bins, labels=labels, right=False)

In [ ]:
# 고위험 건강 지표 생성
df["High_Chol"] = (df["Cholesterol_mg/dL"] >= 240).astype(int)
df["High_BP"] = (df["Blood_Pressure_mmHg"] >= 140).astype(int)
df["High_Glucose"] = (df["Glucose_mg/dL"] >= 126).astype(int)

In [ ]:
# Metabolic_Risk_Score 생성
df["Metabolic_Risk_Score"] = df["High_Chol"] + df["High_BP"] + df["High_Glucose"]

- 건강 상태를 더 잘 나타낼 수 있는 파생 변수들을 생성했습니다. 이러한 피처들은 신경망이 더 효과적으로 패턴을 학습하는 데 도움을 줍니다.

## 4. 입력과 타깃 나누기

- 예측하고자 하는 타깃 변수인 `Diet_Recommendation`을 `y`에 저장하고, 나머지 변수들을 `X`에 저장합니다.
- `Patient_ID`는 식별자일 뿐이므로 제외하고, `Adherence_to_Diet_Plan`은 모든 값이 동일하므로 제거합니다.

In [ ]:
# 타깃 변수(y)와 입력 변수(X) 분리
y = df["Diet_Recommendation"]
X = df.drop(columns=["Diet_Recommendation", "Patient_ID", "Adherence_to_Diet_Plan"])

print(f"입력 데이터 크기: {X.shape}")
print(f"타깃 데이터 크기: {y.shape}")

입력 데이터 크기: (5000, 21)
타깃 데이터 크기: (5000,)


- 입력 데이터와 타깃 데이터를 분리했습니다. 이제 전처리 과정을 거쳐 신경망 모델에 입력할 수 있는 형태로 변환합니다.

## 5. 수치형 / 범주형 데이터 구분

- 신경망 모델에 데이터를 입력하기 전에, 수치형 변수와 범주형 변수를 구분합니다.
- 수치형 변수는 스케일링을 적용하고, 범주형 변수는 원-핫 인코딩을 통해 숫자로 변환합니다.

In [ ]:
# 수치형 피처 리스트 정의
numeric_features = [
    "Age",
    "Weight_kg",
    "Height_cm",
    "BMI",
    "Daily_Caloric_Intake",
    "Weekly_Exercise_Hours",
    "Cholesterol_mg/dL",
    "Blood_Pressure_mmHg",
    "Glucose_mg/dL",
    "Dietary_Nutrient_Imbalance_Score",
    "High_Chol",
    "High_BP",
    "High_Glucose",
    "Metabolic_Risk_Score",
]

In [ ]:
# 범주형 피처 리스트 정의
categorical_features = [
    "Gender",
    "Physical_Activity_Level",
    "Dietary_Restrictions",
    "Allergies",
    "Preferred_Cuisine",
    "BMI_Category",
    "Age_Group",
]

- 수치형 피처 14개와 범주형 피처 7개로 구분했습니다.

## 6. 범주형 데이터 원핫 인코딩

- 범주형 변수를 신경망이 처리할 수 있도록 원핫 인코딩으로 변환합니다.

In [ ]:
# 범주형 변수에 대해 원-핫 인코딩 적용
X_encoded = pd.get_dummies(X, columns=categorical_features)

print(f"원핫 인코딩 후 피처 개수: {X_encoded.shape[1]}")

원핫 인코딩 후 피처 개수: 35


## 7. 학습용 / 검증용 / 테스트용 데이터 분리

- 데이터를 train, validation, test 세 부분으로 나눕니다.
- train은 신경망을 학습하는 데 사용하고, validation은 학습 중 성능을 모니터링하며, test는 최종 성능을 평가하는 데 사용합니다.
- `stratify` 옵션으로 각 데이터셋에서 클래스 비율을 동일하게 유지합니다.

In [ ]:
# train과 temp(validation + test)로 분리 (70% train, 30% temp)
X_train, X_temp, y_train, y_temp = train_test_split(
    X_encoded,
    y,
    test_size=0.3,
    stratify=y,
    random_state=42,
)

# temp를 validation과 test로 분리 (각각 15%씩)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42,
)

print(f"Train 데이터: {X_train.shape}")
print(f"Validation 데이터: {X_val.shape}")
print(f"Test 데이터: {X_test.shape}")

Train 데이터: (3500, 35)
Validation 데이터: (750, 35)
Test 데이터: (750, 35)


- 데이터를 70:15:15 비율로 나누었습니다.

## 8. 수치형 데이터 스케일링

- 신경망은 입력 데이터의 스케일에 민감하므로, StandardScaler로 수치형 변수들을 표준화합니다.
- 평균 0, 표준편차 1로 변환하면 신경망의 학습이 더 안정적이고 빠르게 진행됩니다.
- 스케일러는 반드시 train 데이터로만 fit하고, validation과 test에는 transform만 적용해야 합니다.

In [ ]:
# 스케일러 객체 생성
scaler = StandardScaler()

# Train 데이터로 fit하고 transform
X_train[numeric_features] = scaler.fit_transform(X_train[numeric_features])

# Validation과 Test 데이터는 transform만 적용
X_val[numeric_features] = scaler.transform(X_val[numeric_features])
X_test[numeric_features] = scaler.transform(X_test[numeric_features])

- 수치형 변수들을 표준화했습니다. 이제 모든 피처가 비슷한 스케일을 가지게 되어 신경망 학습에 최적화된 상태가 되었습니다.

## 9. 레이블 인코딩 및 텐서 변환

- PyTorch 모델은 문자열 레이블을 직접 처리할 수 없으므로, LabelEncoder로 타깃을 정수로 변환합니다.
- 예를 들어 'Balanced', 'Low_Carb', 'Low_Sodium'이 0, 1, 2로 인코딩됩니다.

In [ ]:
# 레이블 인코딩
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)
y_test_encoded = label_encoder.transform(y_test)

print(f"클래스 개수: {len(label_encoder.classes_)}")
print(f"클래스 이름: {label_encoder.classes_}")

클래스 개수: 3
클래스 이름: ['Balanced' 'Low_Carb' 'Low_Sodium']


- 이제 입력과 타깃 데이터를 PyTorch 텐서로 변환하여 효율적으로 연산할 수 있게 합니다.
- 입력은 float32, 타깃은 long 타입으로 지정하여 CrossEntropyLoss와 호환되도록 합니다.

In [ ]:
# NumPy 배열을 PyTorch 텐서로 변환
# astype(np.float32)로 명시적으로 float32 타입으로 변환
X_train_tensor = torch.tensor(X_train.to_numpy().astype(np.float32), dtype=torch.float32)
X_val_tensor = torch.tensor(X_val.to_numpy().astype(np.float32), dtype=torch.float32)
X_test_tensor = torch.tensor(X_test.to_numpy().astype(np.float32), dtype=torch.float32)

y_train_tensor = torch.tensor(y_train_encoded, dtype=torch.long)
y_val_tensor = torch.tensor(y_val_encoded, dtype=torch.long)
y_test_tensor = torch.tensor(y_test_encoded, dtype=torch.long)

print(f"X_train shape: {X_train_tensor.shape}")
print(f"y_train shape: {y_train_tensor.shape}")

X_train shape: torch.Size([3500, 35])
y_train shape: torch.Size([3500])


## 10. Dataset과 DataLoader 만들기

- 텐서 데이터를 TensorDataset으로 묶고, DataLoader로 감싸서 배치 단위로 처리합니다.
- DataLoader는 데이터를 자동으로 배치로 나누고, 셔플하며, GPU로 전송하는 작업을 효율적으로 처리합니다.
- batch_size는 한 번에 모델에 입력할 샘플 개수로, 32로 설정합니다.

In [ ]:
# TensorDataset 생성
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [ ]:
# DataLoader 생성
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

Train batches: 110
Validation batches: 24
Test batches: 24


- DataLoader를 생성했습니다.
- train_loader는 매 epoch마다 데이터를 셔플하여 모델이 다양한 패턴을 학습하도록 하고, validation과 test는 셔플하지 않아 일관된 평가가 가능합니다.

## 11. PyTorch 분류 모델 정의하기

- 간단한 3층 신경망을 정의합니다. 입력층 → 은닉층1(64개 뉴런) → 은닉층2(32개 뉴런) → 출력층(클래스 개수) 구조입니다.
- 은닉층에는 ReLU 활성함수를 사용하여 비선형성을 추가하고, Dropout을 사용하여 과적합을 방지합니다.
- 출력층에는 활성함수를 사용하지 않습니다. CrossEntropyLoss가 내부적으로 Softmax를 적용하기 때문입니다.

In [ ]:
# 입력 차원과 클래스 개수 확인
input_dim = X_train_tensor.shape[1]
num_classes = len(label_encoder.classes_)

print(f"입력 차원: {input_dim}")
print(f"클래스 개수: {num_classes}")

입력 차원: 35
클래스 개수: 3


In [ ]:
# 신경망 모델 정의
class DietClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, num_classes)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=0.3)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)
        return x

In [ ]:
# 모델 인스턴스 생성
model = DietClassifier(input_dim=input_dim, num_classes=num_classes)
print(model)

DietClassifier(
  (fc1): Linear(in_features=35, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=32, bias=True)
  (fc3): Linear(in_features=32, out_features=3, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.3, inplace=False)
)


## 12. 손실 함수, 옵티마이저, 디바이스 설정

- 다중 클래스 분류 문제이므로 CrossEntropyLoss를 사용합니다.
- 옵티마이저는 Adam을 사용합니다. Adam은 학습률을 자동으로 조정하여 안정적으로 학습할 수 있게 합니다.
- GPU가 사용 가능하면 GPU에서, 아니면 CPU에서 학습하도록 device를 설정합니다.

In [ ]:
# 손실 함수 정의
criterion = nn.CrossEntropyLoss()

# 옵티마이저 정의
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 디바이스 설정 (GPU 사용 가능 시 GPU, 아니면 CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"사용 디바이스: {device}")

사용 디바이스: cuda


## 13. 모델 학습하기 (Training Loop)

- 신경망 학습은 에폭(epoch) 단위로 진행됩니다. 하나의 에폭은 전체 train 데이터를 한 번 학습하는 것을 의미합니다.
- 각 에폭마다 train과 validation 성능을 확인하여, 모델이 잘 학습되고 있는지, 과적합은 일어나지 않는지 모니터링합니다.
- 매 에폭마다 성능을 출력하여 학습 진행 상황을 확인합니다.

In [ ]:
# 에폭 수 설정
num_epochs = 30

# 학습 시작
for epoch in range(num_epochs):
    # ===== Train 단계 =====
    model.train()  # 모델을 학습 모드로 설정
    train_running_loss = 0.0
    train_correct = 0
    train_total = 0

    for X_batch, y_batch in train_loader:
        # 데이터를 device로 이동
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # 기울기 초기화
        optimizer.zero_grad()

        # Forward pass
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        # Backward pass
        loss.backward()

        # 가중치 업데이트
        optimizer.step()

        # 손실과 정확도 계산
        bs = y_batch.size(0)
        train_running_loss += loss.item() * bs
        pred = outputs.argmax(dim=1)
        train_correct += (pred == y_batch).sum().item()
        train_total += bs

    train_loss = train_running_loss / train_total
    train_acc = train_correct / train_total

    # ===== Validation 단계 =====
    model.eval()  # 모델을 평가 모드로 설정
    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():  # 기울기 계산 비활성화
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            bs = y_batch.size(0)
            val_running_loss += loss.item() * bs
            pred = outputs.argmax(dim=1)
            val_correct += (pred == y_batch).sum().item()
            val_total += bs

    val_loss = val_running_loss / val_total
    val_acc = val_correct / val_total

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
    )

Epoch [1/30] train_loss=0.8794 train_acc=0.5774 val_loss=0.7272 val_acc=0.6760
Epoch [2/30] train_loss=0.6401 train_acc=0.7169 val_loss=0.5396 val_acc=0.7947
Epoch [3/30] train_loss=0.5326 train_acc=0.7763 val_loss=0.4681 val_acc=0.8147
Epoch [4/30] train_loss=0.4721 train_acc=0.8037 val_loss=0.4370 val_acc=0.8187
Epoch [5/30] train_loss=0.4484 train_acc=0.8177 val_loss=0.4150 val_acc=0.8280
Epoch [6/30] train_loss=0.4270 train_acc=0.8251 val_loss=0.4182 val_acc=0.8227
Epoch [7/30] train_loss=0.4146 train_acc=0.8283 val_loss=0.3927 val_acc=0.8373
Epoch [8/30] train_loss=0.4007 train_acc=0.8331 val_loss=0.3866 val_acc=0.8280
Epoch [9/30] train_loss=0.3925 train_acc=0.8351 val_loss=0.3911 val_acc=0.8227
Epoch [10/30] train_loss=0.3833 train_acc=0.8351 val_loss=0.3744 val_acc=0.8333
Epoch [11/30] train_loss=0.3760 train_acc=0.8386 val_loss=0.3699 val_acc=0.8413
Epoch [12/30] train_loss=0.3731 train_acc=0.8394 val_loss=0.3710 val_acc=0.8280
Epoch [13/30] train_loss=0.3630 train_acc=0.8437 

## 14. 테스트 데이터로 모델 평가하기

- 모든 학습이 끝난 후, test 데이터로 최종 성능을 평가합니다.
- test 데이터는 학습 과정에서 한 번도 사용되지 않았으므로, 모델의 일반화 성능을 측정할 수 있습니다.
- 손실과 정확도를 계산하여 모델이 얼마나 잘 작동하는지 확인합니다.

In [ ]:
model.eval()

test_loss_sum = 0.0
test_correct = 0
test_total = 0

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        bs = y_batch.size(0)
        test_loss_sum += loss.item() * bs
        pred = outputs.argmax(dim=1)
        test_correct += (pred == y_batch).sum().item()
        test_total += bs

test_loss = test_loss_sum / test_total
test_acc = test_correct / test_total

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

Test Loss: 0.3761
Test Accuracy: 0.8320


## 15. 다양한 평가 지표 확인하기

- classification_report를 출력하여 각 클래스별 precision, recall, f1-score를 확인합니다.
- 어떤 식단 유형을 잘 예측하는지, 어떤 유형에서 어려움을 겪는지 분석할 수 있습니다.
- 모델을 평가 모드로 설정하고, test 데이터에 대한 예측을 수행하여 각 클래스별 성능을 분석합니다.

In [ ]:
# 모델을 평가 모드로 설정
model.eval()

test_preds = []
test_targets = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        outputs = model(X_batch)
        pred = outputs.argmax(dim=1)

        test_preds.extend(pred.cpu().tolist())
        test_targets.extend(y_batch.cpu().tolist())

# Classification Report 출력
print("=== Classification Report ===")
print(classification_report(test_targets, test_preds, target_names=label_encoder.classes_))

=== Classification Report ===
              precision    recall  f1-score   support

    Balanced       0.83      0.91      0.87       419
    Low_Carb       0.87      0.82      0.85       147
  Low_Sodium       0.79      0.66      0.72       184

    accuracy                           0.83       750
   macro avg       0.83      0.80      0.81       750
weighted avg       0.83      0.83      0.83       750

